### RAG Pipeline - Data Ingestion to vector DB Pileline

In [1]:
## importing necessary library

import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

c:\Users\KAPIL\OneDrive\Documents\Pictures\Desktop\Genrative AI\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Read all the pdf's inside the directory

def process_all_pdfs(pdf_directory):
    """Process all pdf files in the given directory"""
    pdf_dir=Path(pdf_directory)
    all_documents=[]
    
    # Find all pdf files in the directory
    pdf_files=list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files in the directory.")
    
    for pdf_file in pdf_files:
        print(f"\nprocessing file: {pdf_file.name}")
        #Load the pdf file
        try:
            loader=PyPDFLoader(str(pdf_file))
            documents=loader.load()
            
            # Add source info to the metadata
            for doc in documents:
                doc.metadata['source']=pdf_file.name
                doc.metadata['file_type']="pdf"
                
            all_documents.extend(documents)
            print(f"Loaded {len(documents)} documents from {pdf_file.name}")
            
        except Exception as e:
            print(f"Error loading {pdf_file.name}: {e}")
            
    print(f"\nTotal documents loaded: {len(all_documents)}")
    
    return all_documents

all_pdf_documents=process_all_pdfs("../data/pdfFiles")

Found 2 PDF files in the directory.

processing file: DBMS_Notes.pdf
Loaded 49 documents from DBMS_Notes.pdf

processing file: OS_Full_Notes.pdf
Loaded 41 documents from OS_Full_Notes.pdf

Total documents loaded: 90


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'Acrobat Pro 22.3.20258', 'creator': 'Acrobat Pro 22.3.20258', 'creationdate': '2022-10-23T00:01:18+05:30', 'moddate': '2022-10-23T00:01:18+05:30', 'title': '', 'source': 'DBMS_Notes.pdf', 'total_pages': 49, 'page': 0, 'page_label': '1', 'file_type': 'pdf'}, page_content='LEC-1: Introduction to DBMS \n1. What is Data?\na. Data is a collection of raw, unorganized facts and details like text, observations, figures, symbols,\nand descriptions of things etc.\nIn other words, data does not carry any specific purpose and has no significance by itself.\nMoreover, data is measured in terms of bits and bytes – which are basic units of information in the\ncontext of computer storage and processing.\nb. Data can be recorded and doesn’t have any meaning unless processed.\n2. Types of Data\na. Quantitative\ni. Numerical form\nii. Weight, volume, cost of an item.\nb. Qualitative\ni. Descriptive, but not numerical.\nii. Name, gender, hair color of a person.\n3. What is

In [4]:
### Text Splitting and chunking the documents

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks"""
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    
    split_docs=text_splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_docs)} chunks.")
    
    # Show an Example of chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [5]:
chunks=split_documents(all_pdf_documents)
chunks

split 90 documents into 214 chunks.

Example chunk:
Content: LEC-1: Introduction to DBMS 
1. What is Data?
a. Data is a collection of raw, unorganized facts and details like text, observations, figures, symbols,
and descriptions of things etc.
In other words, d...
Metadata: {'producer': 'Acrobat Pro 22.3.20258', 'creator': 'Acrobat Pro 22.3.20258', 'creationdate': '2022-10-23T00:01:18+05:30', 'moddate': '2022-10-23T00:01:18+05:30', 'title': '', 'source': 'DBMS_Notes.pdf', 'total_pages': 49, 'page': 0, 'page_label': '1', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Acrobat Pro 22.3.20258', 'creator': 'Acrobat Pro 22.3.20258', 'creationdate': '2022-10-23T00:01:18+05:30', 'moddate': '2022-10-23T00:01:18+05:30', 'title': '', 'source': 'DBMS_Notes.pdf', 'total_pages': 49, 'page': 0, 'page_label': '1', 'file_type': 'pdf'}, page_content='LEC-1: Introduction to DBMS \n1. What is Data?\na. Data is a collection of raw, unorganized facts and details like text, observations, figures, symbols,\nand descriptions of things etc.\nIn other words, data does not carry any specific purpose and has no significance by itself.\nMoreover, data is measured in terms of bits and bytes – which are basic units of information in the\ncontext of computer storage and processing.\nb. Data can be recorded and doesn’t have any meaning unless processed.\n2. Types of Data\na. Quantitative\ni. Numerical form\nii. Weight, volume, cost of an item.\nb. Qualitative\ni. Descriptive, but not numerical.\nii. Name, gender, hair color of a person.\n3. What is

### Embedding and Vectore Store DB


In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        Args:
            model_name: HuggingFace model name for sentence embeddings (dimension: 384 for all-MiniLM-L6-v2)
        """
        self.model_name = model_name
        self.model = None
        self._load_model()
        
    
    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
        
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        Args:
            texts: List of text strings to embed
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2521.06it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


### Vector Store

In [8]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore


Vector store initialized. Collection: pdf_documents
Existing documents in collection: 1070


In [9]:
chunks

[Document(metadata={'producer': 'Acrobat Pro 22.3.20258', 'creator': 'Acrobat Pro 22.3.20258', 'creationdate': '2022-10-23T00:01:18+05:30', 'moddate': '2022-10-23T00:01:18+05:30', 'title': '', 'source': 'DBMS_Notes.pdf', 'total_pages': 49, 'page': 0, 'page_label': '1', 'file_type': 'pdf'}, page_content='LEC-1: Introduction to DBMS \n1. What is Data?\na. Data is a collection of raw, unorganized facts and details like text, observations, figures, symbols,\nand descriptions of things etc.\nIn other words, data does not carry any specific purpose and has no significance by itself.\nMoreover, data is measured in terms of bits and bytes – which are basic units of information in the\ncontext of computer storage and processing.\nb. Data can be recorded and doesn’t have any meaning unless processed.\n2. Types of Data\na. Quantitative\ni. Numerical form\nii. Weight, volume, cost of an item.\nb. Qualitative\ni. Descriptive, but not numerical.\nii. Name, gender, hair color of a person.\n3. What is

In [10]:
### Convert the text to embeddings

## Extract the text content from the chunks
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings
print(f"Generating embeddings for {len(texts)} chunks...")
embeddings=embedding_manager.generate_embeddings(texts)

## store in the vector database
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 214 chunks...
Generating embeddings for 214 texts...


Batches: 100%|██████████| 7/7 [00:11<00:00,  1.58s/it]


Generated embeddings with shape: (214, 384)
Adding 214 documents to vector store...
Successfully added 214 documents to vector store
Total documents in collection: 1284


### Retriever Pipeline From VectorStore

In [11]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)
rag_retriever

In [12]:
rag_retriever.retrieve("What is data")

Retrieving documents for query: 'What is data'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 28.26it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_70f0d535_0',
  'content': 'LEC-1: Introduction to DBMS \n1. What is Data?\na. Data is a collection of raw, unorganized facts and details like text, observations, figures, symbols,\nand descriptions of things etc.\nIn other words, data does not carry any specific purpose and has no significance by itself.\nMoreover, data is measured in terms of bits and bytes – which are basic units of information in the\ncontext of computer storage and processing.\nb. Data can be recorded and doesn’t have any meaning unless processed.\n2. Types of Data\na. Quantitative\ni. Numerical form\nii. Weight, volume, cost of an item.\nb. Qualitative\ni. Descriptive, but not numerical.\nii. Name, gender, hair color of a person.\n3. What is Information?\na. Info. Is processed, organized, and structured data.\nb. It provides context of the data and enables decision making.\nc. Processed data that make sense to us.\nd. Information is extracted from the data, by analyzing and interpreting pieces of data

### RAG Pipeline: VectorDB To LLM Output Generation

In [13]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.3-70b-versatile",temperature=0.7, max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question
        Context:
        {context}
        Question: {query}
        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content



In [14]:
# Question in COntext

answer=rag_simple("What is Thread Scheduling", rag_retriever, llm)
print("Answer:", answer)

Retrieving documents for query: 'What is Thread Scheduling'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 39.00it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: Thread Scheduling refers to the process by which threads are scheduled for execution based on their priority. The operating system assigns processor time slices to all threads, allowing them to execute within their runtime. This means that the threads are given a specific amount of time to run before the operating system switches to another thread, ensuring that all threads have a chance to execute and making concurrent execution possible.


In [15]:
#Question outof context

answer=rag_simple("Who is Kapil Gupta",rag_retriever,llm)
print("Answer:", answer)

Retrieving documents for query: 'Who is Kapil Gupta'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 37.83it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
Answer: No relevant context found to answer the question.


### Enhanced RAG Pipleine

In [16]:
# --- Enhanced RAG Pipeline Features ---

def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results=retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output


In [17]:
# Example usage of the advanced RAG function:
result = rag_advanced("What are Threads?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

print("\n---\n")

result2=rag_advanced("Who is Kapil Gupta?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result2['answer'])
print("Sources:", result2['sources'])
print("Confidence:", result2['confidence'])
print("Context Preview:", result2['context'][:300])


Retrieving documents for query: 'What are Threads?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 63.03it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: Threads are a single sequence stream within a process, an independent path of execution, and a light-weight process, used to achieve parallelism by dividing a process's tasks into independent paths of execution.
Sources: [{'source': 'OS_Full_Notes.pdf', 'page': 20, 'score': 0.1359599232673645, 'preview': 'LEC-15: Introduction to Concurrency \n1. Concurrency is the execution of the multiple instruction sequences at the same time. It happens in\nthe operating system when there are several process threads running in parallel.\n2. Thread:\n• Single sequence stream within a process.\n• An independent path of e...'}, {'source': 'OS_Full_Notes.pdf', 'page': 20, 'score': 0.1359599232673645, 'preview': 'LEC-15: Introduction to Concurrency \n1. Concurrency is the execution of the multiple instruction sequences at the same time. It happens in\nthe operating system when there are several process threads running in parallel.\n2. Thread:\n• Single sequence stream within a process.\n• An inde

Batches: 100%|██████████| 1/1 [00:00<00:00, 94.88it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
Answer: No relevant context found.
Sources: []
Confidence: 0.0
Context Preview: 


In [19]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("What is kernel?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'What is kernel?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 32.44it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
LEC-4: Components	of	OS
1.Kernel:	A	kernel	is	that	part	of	the operating	system	which	interacts	directly with	
the	hardware	and																																										pe

rforms	the	most crucial	tasks.
a.Heart	of	OS/Core	component
b.Very	first	part	of	OS	to	load	on start-up.
2.User	space:	Where	application	software runs,	apps	don’t	have	privileged	access	to the	
underlying	hardware.	It	interacts	with kernel.
a.GUI
b.CLI
A	shell,	also	known	as	a	command	interpreter,	is	that	part	of	the	operating	system	that	receives	
commands	from	the	users	and	gets	them	executed.	
Functions	of	Kernel:	
1.Process	management:
a.Scheduling	processes	and	threads	on	the	CPUs.
b.Creating	&	deleting	both	user	and	system	process.
c.Suspending	and	resuming	processes
d.Providing	mechanisms	for	process	synchronization	or	process 
communication.
2.Memory	management:
a.Allocating	and	deallocating	memory	space	as	per	need.
b.Keeping	track	of	which	part	of	memory	are	currently	being	used	and	by 
which	process.

LEC-4: Components	of	OS
1.Kernel:	A	kernel	is	that	part	of	the operating	system	which	interacts	directly with	
the	hardware	and																																										perform